# Análise CEAPS #7DaysOfCode

Nesse projeto, vamos realizar a análise exploratória e tratamento de dados dos últimos quatro anos Cota para Exercício da Atividade Parlamentar dos Senadores (CEAPS), durante o desafio 7 Days of Code.

Dados brutos disponíveis em https://www12.senado.leg.br/transparencia/dados-abertos-transparencia/dados-abertos-ceaps?utm_medium=email&_hsenc=p2ANqtz-9mPVAQsNoeBFn9A6lRxaaB05EFC1Z8LtuTmLhT3jedf2-6jKUefIOT4ztFbmtrqolX8Q4pBchyRFRHNPOoV3ArDlg7tA&_hsmi=231298145&utm_content=231298145&utm_source=hs_automation

In [15]:
import pandas as pd
import matplotlib.pyplot as plt 
import numpy as np
import os

In [16]:
# Carregar e concatenar os dados dos últimos 4 anos - Load and concatenate the last 4 years of data
# versão 07/Maio/2026: raw_data contém os arquivos de 2008 a 2022; selecionamos os últimos 4 anos (2019-2022) para análise - version 07/May/2026: raw_data contains files from 2008 to 2022; we select the last 4 years (2019-2022) for analysis

csv_files = sorted([f for f in os.listdir("raw_data") if f.lower().endswith(".csv")])
selected_files = csv_files[-4:]
data_dir = os.path.join(os.getcwd(), "raw_data")
header = pd.read_csv(os.path.join(data_dir, selected_files[0]),sep=";",encoding="latin1",skiprows=1,nrows=0)
dfs = [pd.read_csv(os.path.join(data_dir, file),sep=";",encoding="latin1",skiprows=[0]) for file in selected_files]
df_final = pd.concat(dfs, ignore_index=True)
df_final.head()

,ANO,MES,SENADOR,TIPO_DESPESA,CNPJ_CPF,FORNECEDOR,DOCUMENTO,DATA,DETALHAMENTO,VALOR_REEMBOLSADO,COD_DOCUMENTO
0,2019,1,ACIR GURGACZ,"Aluguel de imóveis para escritório político, c...",05.914.650/0001-66,ENERGISA,006582758,04/01/2019,Despesa com pagamento de energia elétrica do e...,"66,02",2116543
1,2019,1,ACIR GURGACZ,"Aluguel de imóveis para escritório político, c...",05.914.650/0001-66,ENERGISA,006582755,04/01/2019,Despesa com pagamento de energia elétrica do e...,"139,98",2116546
2,2019,1,ACIR GURGACZ,"Aluguel de imóveis para escritório político, c...",004.948.028-63,GILBERTO PISELO DO NASCIMENTO,00119,07/01/2019,Despesa com pagamento de aluguel de imóvel par...,6000,2113817
3,2019,1,ACIR GURGACZ,"Aluguel de imóveis para escritório político, c...",05.423.963/0001-11,OI MÓVEL S.A.,86161151,25/12/2018,Despesa com pagamento de telefonia para o escr...,"316,39",2116541
4,2019,2,ACIR GURGACZ,"Aluguel de imóveis para escritório político, c...",05.914.650/0001-66,ENERGISA,007236036,04/02/2019,Despesa com pagamento de energia elétrica para...,"99,45",2116550


In [17]:
# Limpando os dados - Cleaning the data
# 1. checando os tipos de dados de cada coluna - checking the data types of each column
print(df_final.dtypes)

ANO                  int64
MES                  int64
SENADOR                str
TIPO_DESPESA           str
CNPJ_CPF               str
FORNECEDOR             str
DOCUMENTO              str
DATA                   str
DETALHAMENTO           str
VALOR_REEMBOLSADO      str
COD_DOCUMENTO        int64
dtype: object


In [18]:
# Corrijir o formato da coluna "CNPJ_CPF" - Correct the format of the "CNPJ_CPF" column
df_final["CNPJ_CPF"] = (
    df_final["CNPJ_CPF"]
    .astype(str)
    .str.replace(r"\D", "", regex=True)  # remove pontos, barras, hífens etc.
    .str.zfill(14)                       # completa com zeros à esquerda
)

# Corrijir o formato da data na coluna "DATA" - Correct the date format in the "DATA" column
df_final["DATA"] = pd.to_datetime(
    df_final["DATA"],
    dayfirst=True,
    errors="coerce")

# Corrijir o formato da coluna "VALOR_REEMBOLSADO" - Correct the format of the "VALOR_REEMBOLSADO" column
df_final["VALOR_REEMBOLSADO"] = (
    df_final["VALOR_REEMBOLSADO"]
    .astype(str)
    .str.replace(",", ".", regex=False)  # troca vírgula decimal
    .astype(float)
)

# df_final.head()

In [22]:
# 2. checking for missing values
# print(df_final.isnull().sum())

# remover coluna 'DETALHAMENTO' pois possui muitos valores ausentes e não traz muita informação - drop column 'DETALHAMENTO' since it has many missing values and doesn't provide much information
df_final.drop(columns=["DETALHAMENTO"], inplace=True)

# preencher os valores ausentes na coluna 'DOCUMENTO' - fill missing values in 'DOCUMENTO'
df_final["DOCUMENTO"] = df_final["DOCUMENTO"].fillna(0)

In [20]:
# 3. removendo duplicatas - removing duplicates
df_final.drop_duplicates(inplace=True)

In [23]:
df_final.head()

,ANO,MES,SENADOR,TIPO_DESPESA,CNPJ_CPF,FORNECEDOR,DOCUMENTO,DATA,VALOR_REEMBOLSADO,COD_DOCUMENTO
0,2019,1,ACIR GURGACZ,"Aluguel de imóveis para escritório político, c...",05914650000166,ENERGISA,006582758,2019-01-04,66.02,2116543
1,2019,1,ACIR GURGACZ,"Aluguel de imóveis para escritório político, c...",05914650000166,ENERGISA,006582755,2019-01-04,139.98,2116546
2,2019,1,ACIR GURGACZ,"Aluguel de imóveis para escritório político, c...",00000494802863,GILBERTO PISELO DO NASCIMENTO,00119,2019-01-07,6000.00,2113817
3,2019,1,ACIR GURGACZ,"Aluguel de imóveis para escritório político, c...",05423963000111,OI MÓVEL S.A.,86161151,2018-12-25,316.39,2116541
4,2019,2,ACIR GURGACZ,"Aluguel de imóveis para escritório político, c...",05914650000166,ENERGISA,007236036,2019-02-04,99.45,2116550
